# QASP optimization algorithms

Executable smoke tests for standard and multi-angle QAOA, custom mixers, both warm-start variants, recursive QAOA, LR-QAOA, PCE, shot-based training and sampling, circuit drawing, PyTorch optimizers, and PennyLane-native optimizers.


In [ ]:
import numpy as np
from qasp.optimization import (
    LRQAOA, PCE, QAOA, QUBO, RecursiveQAOA, WarmStartQAOA,
    draw_optimization,
)

problem = QUBO.maxcut(3, [(0, 1), (1, 2), (0, 2)])
exact_bitstring, exact_energy = problem.brute_force()
assert exact_energy == -2.0
exact_bitstring, exact_energy

In [ ]:
# Standard QAOA and a manually specified QASP mixer dictionary
custom_mixer = {
    'rotations': ['X 0', 'X 1', 'X 2', 'XX 0 1'],
    'initial_state': 'plus',
}
standard = QAOA(problem, n_layers=1, mixer=custom_mixer, seed=1)
standard_result = standard.solve(optimizer_options={'maxiter': 20})
assert standard.parameter_shape == (1, 2)
assert np.isclose(standard_result.probabilities.sum(), 1.0)
standard_result

In [ ]:
# Multi-angle QAOA: one angle per cost and mixer gate
multi = QAOA(problem, n_layers=1, parametrization='multi_angle', seed=2)
multi_result = multi.solve(optimizer_options={'maxiter': 20})
assert multi.parameter_shape == (1, len(multi.cost_terms) + len(multi.mixer_terms))
assert multi_result.algorithm == 'multi_angle_qaoa'
multi_result

In [ ]:
# Warm-start works with standard and multi-angle parameterizations
warm_solvers = {}
warm_results = {}
for mode in ('standard', 'multi_angle'):
    warm_solvers[mode] = WarmStartQAOA(
        problem, n_layers=1, parametrization=mode, warm_start=[0.2, 0.8, 0.2]
    )
    warm_results[mode] = warm_solvers[mode].solve(
        initial_parameters='zeros', optimizer_options={'maxiter': 8}
    )
    assert 'warm_start' in warm_results[mode].algorithm
warm_results

In [ ]:
# RQAOA eliminates one variable, solves at the cutoff, then reconstructs it
recursive_solver = RecursiveQAOA(
    problem, cutoff=2, optimizer_options={'maxiter': 10}
)
recursive_result = recursive_solver.solve()
assert len(recursive_result.elimination_rules) == 1
assert recursive_result.energy == exact_energy
recursive_result

In [ ]:
# LR-QAOA uses the deterministic opposing linear ramps, without optimization
linear_ramp = LRQAOA(problem, n_layers=4, delta_gamma=0.6, delta_beta=0.3)
lr_result = linear_ramp.solve()
np.testing.assert_allclose(linear_ramp.ramp_parameters[0], [0.15, 0.3])
np.testing.assert_allclose(linear_ramp.ramp_parameters[-1], [0.6, 0.075])
lr_result

In [ ]:
# PCE with a QASP AngleAnsatz supplied using the package's dictionary convention
pce_circuit = {
    'ansatz': 'AngleAnsatz',
    'ansatz_config': {
        'n_layers': 1,
        'rotations': ['Y 0', 'Y 1', 'ZZ 0 1'],
        'include_hadamards_initial': True,
    },
    'device': 'default.qubit',
}
pce = PCE(problem, circuit=pce_circuit, seed=4)
pce_result = pce.solve(optimizer_options={'maxiter': 15})
assert pce.n_qubits == 2
assert pce.pauli_labels == ['XX 0 1', 'YY 0 1', 'ZZ 0 1']
assert len(pce_result.bitstring) == problem.n_variables
pce_result

In [ ]:
# The cost is a Hamiltonian expectation: adjoint gradients analytically,
# or finite-shot training and independently sampled final solutions.
adjoint_gradient = standard.gradient(standard_result.parameters)
assert standard.diff_method == 'adjoint'
assert np.isfinite(adjoint_gradient).all()

shot_solver = QAOA(problem, n_layers=1, shots=200, seed=8)
shot_result = shot_solver.solve(
    initial_parameters='zeros',
    optimizer='COBYLA',
    optimizer_options={'maxiter': 5},
    result_shots=500,
)
assert shot_solver.diff_method == 'parameter-shift'
assert sum(shot_result.counts.values()) == 500
assert np.isclose(shot_result.solution_probability,
                  shot_result.counts[shot_result.bitstring] / 500)
shot_result

## NumPy/PennyLane and PyTorch optimizer backends

The NumPy backend supports SciPy and PennyLane optimizers, including QNG, SPSA, QNSPSA, and ShotAdaptive. The PyTorch backend exposes Torch optimizers such as Adam, AdamW, SGD, RMSprop, Adagrad, Adadelta, ASGD, and LBFGS.


In [ ]:
from qasp.optimization import PENNYLANE_OPTIMIZERS, TORCH_OPTIMIZERS

torch_solver = QAOA(problem, n_layers=1, backend='torch', seed=8)
torch_result = torch_solver.solve(
    optimizer='Adam', optimizer_options={'max_steps': 2, 'lr': 0.05}
)
assert torch_result.metadata['backend'] == 'torch'
assert torch_result.metadata['optimizer'] == 'Adam'

qng_solver = QAOA(problem, n_layers=1, backend='numpy', seed=9)
qng_result = qng_solver.solve(
    optimizer='QNG',
    optimizer_options={'max_steps': 1, 'stepsize': 0.02, 'lam': 1e-3},
)
assert qng_result.metadata['optimizer'] == 'QNG'

spsa_solver = QAOA(problem, shots=100, seed=10)
spsa_result = spsa_solver.solve(
    optimizer='SPSA',
    optimizer_options={'max_steps': 1, 'a': 0.02, 'c': 0.1},
)
assert spsa_result.metadata['optimizer'] == 'SPSA'

pce_torch = PCE(problem, circuit=pce_circuit, backend='torch', seed=11)
pce_torch_result = pce_torch.solve(
    optimizer='Adam', optimizer_options={'max_steps': 1, 'lr': 0.02}
)
assert np.isfinite(pce_torch_result.objective)
assert {'QNG', 'SPSA', 'QNSPSA'} <= set(PENNYLANE_OPTIMIZERS)
assert {'Adam', 'AdamW', 'SGD', 'RMSprop'} <= set(TORCH_OPTIMIZERS)

{
    'torch_adam': torch_result.objective,
    'pennylane_qng': qng_result.objective,
    'pennylane_spsa': spsa_result.objective,
    'pce_torch': pce_torch_result.objective,
}


In [ ]:
# Every optimization circuit supports symbolic, standalone Quantikz/TikZ.
paper_circuits = {
    'standard': draw_optimization(
        standard, parameters=standard_result, output='tikz',
        tikz_options={'standalone': True},
    ),
    'multi-angle': multi.draw(multi_result, output='tikz'),
    'warm-start': warm_solvers['multi_angle'].draw(
        warm_results['multi_angle'], output='tikz'
    ),
    'recursive-step': recursive_solver.draw(step=0, output='tikz'),
    'linear-ramp': linear_ramp.draw(output='tikz'),
    'PCE': pce.draw(pce_result, output='tikz'),
}
for name, source in paper_circuits.items():
    assert '\\begin{quantikz}' in source
    assert '\\meter' in source
assert '\\gamma_{1}' in paper_circuits['standard']
assert '\\gamma_{1,1}' in paper_circuits['multi-angle']
print({name: len(source) for name, source in paper_circuits.items()})

In [ ]:
summary = {
    'standard QAOA': standard_result.energy,
    'multi-angle QAOA': multi_result.energy,
    'warm-start QAOA': warm_results['standard'].energy,
    'warm-start multi-angle QAOA': warm_results['multi_angle'].energy,
    'recursive QAOA': recursive_result.energy,
    'LR-QAOA': lr_result.energy,
    'PCE': pce_result.energy,
}
print('All optimization notebook tests passed.')
summary